# Simulated annealing & tabu search — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Escape local traps by sometimes accepting worse moves or forbidding immediate backtracking
Metaheuristics are disciplined ways to keep search from getting stuck. These five examples show Metropolis acceptance, temperature cooling, simulated annealing escaping a local basin, tabu memory blocking a reversal, and tabu tenure changing the search path.

In [ ]:
# 1) Metropolis acceptance for uphill moves
Delta=2.; T=1.; prob=math.exp(-Delta/T); plt.figure(figsize=(5,3)); plt.bar(['accept uphill'],[prob]); plt.ylim(0,1); plt.title('worse moves are possible, not free')
assert abs(prob-0.1353352832366127)<1e-12

In [ ]:
# 2) cooling lowers uphill acceptance
Ts=np.array([5.,2.,1.,0.5,0.1]); probs=np.exp(-Delta/Ts); plt.figure(figsize=(5,3)); plt.plot(Ts,probs,'-o'); plt.xlabel('temperature'); plt.ylabel('P(accept worse)'); plt.title('cooling becomes greedier')
assert probs[-1]<probs[0]

In [ ]:
# 3) annealing can escape a local basin
rng=np.random.default_rng(0); xs=np.linspace(-3,3,400); f=lambda x:0.15*x**4-x**2+0.2*x; x=1.5; path=[]
for t in range(200):
    T=max(0.05,2*(0.97**t)); cand=x+rng.normal(0,0.4); d=f(cand)-f(x)
    if d<0 or rng.random()<math.exp(-d/T): x=cand
    path.append(x)
plt.figure(figsize=(5,3)); plt.plot(xs,f(xs)); plt.plot(path,[f(v) for v in path],alpha=.5); plt.title('annealed trajectory')
assert f(x)<f(1.5)

In [ ]:
# 4) tabu memory prevents immediate reversal
state=0; moves=[1,-1,1]; tabu=[]; accepted=[]
for m in moves:
    if -m in tabu: accepted.append(False)
    else: state+=m; accepted.append(True); tabu=[m]
plt.figure(figsize=(5,3)); plt.bar(['+1','-1','+1 again'],[1 if a else 0 for a in accepted]); plt.title('reverse move is tabu')
assert accepted==[True,False,True]

In [ ]:
# 5) longer tabu tenure changes visited states
moves=[-2,-1,2];
def run(tenure):
    state=0; tabu=[]; path=[state]
    for m in moves:
        if -m not in tabu:
            state+=m; tabu.append(m); tabu=tabu[-tenure:]
        path.append(state)
    return path
p1=run(1); p2=run(2); plt.figure(figsize=(5,3)); plt.plot(p1,'-o',label='tenure 1'); plt.plot(p2,'-s',label='tenure 2'); plt.legend(); plt.title('memory length changes search')
assert p1!=p2